In [1]:
import pandas as pd
import fastembed 
import qdrant_client

# FASE 1 INDICIZZAZIONE

In [95]:
import qdrant_client
from qdrant_client import models
client = qdrant_client.QdrantClient('http://localhost:6333', timeout=1000)

In [96]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("jinaai/jina-embeddings-v3", cache_dir = './fastembed/')
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25", cache_dir = './fastembed/')
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0", cache_dir = './fastembed/')

In [97]:
client.create_collection(
            collection_name='rag_lezioni',
            vectors_config={
                "dense": models.VectorParams(
                    size=1024,
                    distance=models.Distance.COSINE
                ),
                 "colbert": models.VectorParams(
                size=128,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
                hnsw_config=models.HnswConfigDiff(m=0)  # Disable HNSW for reranking
        )
                
            },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [68]:
df_db = pd.read_excel('../data/V2_elenco-contenuti-editori_AGGIORNATI_da-backend.xlsx')

In [69]:
df_db.head()

,esId,Link,Editore,Titolo,Descrizione,Tipo,Discipline,Ordine di scuola min,Ordine di scuola max,LIV MIN = LIV MAX,...,Visualizzazioni,Vis. Docenti,Vis. Studenti,Utilizzi,Segnalibri,Like,type,id,data,_score
0,chVmZZUBUNuIiffIyHKc,abb/prova_html.zip/index.html,ABB Italia,DA ELIMINARE,Prova html,approfondimenti,Italiano,0.0,13.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,uIituYgBfcwcz2xFZf7v,abb/lockdown.mp4/lockdown.mp4,ABB Italia,DA ELIMINARE,video,video,Italiano,0.0,13.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,mbrv9oABmJEGg9qZpp1W,https://www.youtube.com/watch?v=mwz6DT4yncE,ABB Italia,"Decarbonizzazione, transizione energetica ed e...","Enrico Ragaini, Senior Principal Engineer, e P...",video,"Economia,Cittadinanza e Costituzione",11.0,12.0,0.0,...,0.0,0.0,0.0,4.0,6.0,1.0,NaN,NaN,NaN,NaN
3,b8IW14ABeWek0aUchcGn,abb/mini-impreseperlambienteelosviluppososteni...,ABB Italia,BUSINESS KIT | Mini-imprese per le aziende e l...,Presentazione in pdf che illustra il concetto ...,"approfondimenti,tools","Economia,Cittadinanza e Costituzione",10.0,12.0,0.0,...,46.0,20.0,3.0,23.0,6.0,0.0,NaN,NaN,NaN,NaN
4,OqiK24ABmJEGg9qZA0id,abb/podcast_marketing_manager_-_powered_by_abb...,ABB Italia,BUSINESS KIT | Marketing manager,In questo podcast by ABB approfondiremo il ruo...,audio_podcast,"Cittadinanza e Costituzione,Economia",10.0,11.0,0.0,...,4.0,1.0,0.0,96.0,9.0,0.0,NaN,NaN,NaN,NaN


In [98]:
df_filo = df_db.loc[df_true.Discipline.str.lower().str.contains('filosofia')]

In [101]:
df_filo.loc[:, 'Ordine di scuola max'] = df_filo['Ordine di scuola max'].apply(lambda x: min(x,12))

In [102]:
df_filo_random = df_filo.sample(frac=1.0, random_state=0).reset_index(drop=True)

In [103]:
df_test = df_filo_random.iloc[:469]
df_test.to_csv('df_test_filosofia.csv')
df_inference = df_filo_random.iloc[469:]

In [104]:
disciplina = list(df_inference.Discipline)
liv_min = list(df_inference['Ordine di scuola min'])
liv_max = list(df_inference['Ordine di scuola max'])
titolo = list(df_inference['Titolo'])
descrizione = list(df_inference.Descrizione)


1000

In [105]:
dense_embeddings = list(dense_embedding_model.embed(text for text in descrizione))

In [106]:
late_interaction_embeddings = list(late_interaction_embedding_model.embed(text for text in descrizione))

In [107]:
bm25_embeddings = list(bm25_embedding_model.embed(text for text in descrizione))

In [108]:
from qdrant_client.models import PointStruct

points = []
for indice,(dense_embedding, bm25_embedding, late_interaction_embedding, disc, tit, liv_minimo, liv_maximo, descr) in enumerate(zip(dense_embeddings, bm25_embeddings, late_interaction_embeddings, disciplina, titolo, liv_min, liv_max,  descrizione)):
  
    point = PointStruct(
        id=indice,
        vector={
            "dense": dense_embedding,
            "colbert": late_interaction_embedding,
            "bm25": bm25_embedding.as_object(),
        },
        payload={"descr": descr, "disciplina": disc, "liv_min": liv_minimo, "liv_max": liv_maximo, "titolo": tit}
    )
    points.append(point)

In [109]:
for batch in range(len(points) // 20):
    operation_info = client.upsert(
    collection_name="rag_lezioni",
    points=points[batch * 20:(batch+1) * 20]
)
operation_info = client.upsert(
    collection_name="rag_lezioni",
    points=points[batch * 20:])
print(operation_info)

operation_id=51 status=<UpdateStatus.COMPLETED: 'completed'>


# FASE 2: FASE QUERY

In [32]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [33]:
import torch
torch.device('cuda' if torch.cuda.is_available() else 'cpu')


device(type='cuda')

In [34]:
model.to('cuda')

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [35]:
#query = "Si descrive Platone e il mondo delle idee"
query = "Il prof. Matteo Saudino, ideatore del canale YouTube <i>BarbaSophia</i>, legge un passaggio del Manifesto ed Engels in cui si desrivono il ruolo e la trasformazione della borghesia."

In [36]:
dense_vector = next(dense_embedding_model.query_embed(query))
sparse_vector = next(bm25_embedding_model.query_embed(query))
late_interaction_vector = next(late_interaction_embedding_model.query_embed(query))

In [11]:
prefetch = [
        models.Prefetch(
            query=dense_vector,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_vector.as_object()),
            using="bm25",
            limit=1,
        ),
    ]

In [12]:
results = client.query_points(
         "rag_capitoli",
        prefetch=prefetch,
        query=late_interaction_vector,
        using="colbert",
        with_payload=True,
        limit=20,
)

points = results.points

In [13]:
#model.to('cuda')
results = model.rerank(query, [point.payload['descr'] for point in points])

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

Score: 0.1647
Document: Marx e il materialismo storico
IL SOCIALISMO SCIENTIFICO
Il sostegno economico di Engels
L'amicizia ...

Score: 0.1032
Document: Marx e il materialismo storico
Il Manifesto del partito comunista

Presenta il "Manifesto del partit...

Score: 0.0013
Document: Marx e il materialismo storico
Economia borghese e alienazione
I quattro aspetti dell’alienazione
De...

Score: -0.0083
Document: Marx e il materialismo storico
Economia borghese e alienazione

Esamina la critica di Marx all'econo...

Score: -0.0634
Document: Marx e il materialismo storico
Il sogno di Marx
La dittatura del proletariato
Spiega la fase transit...

Score: -0.0635
Document: Marx e il materialismo storico
Il Manifesto del partito comunista
Il socialismo scientifico
Spiega i...

Score: -0.0695
Document: Marx e il materialismo storico
IL SOCIALISMO SCIENTIFICO
Il mondo di Marx e Engels
Lo scenario della...

Score: -0.0772
Document: Marx e il materialismo storico
Il Manifesto del partito comunista
Ri

In [ ]:
results = [point.payload for point in points if point.payload['descr'] in [result['document'] for result in results]]

In [133]:
SYSTEM_MESSAGE = """# SYSTEM MESSAGE: ANALISTA PEDAGOGICO

Sei un esperto di sistemi educativi. Il tuo compito è determinare il target scolare (0-12) di una lezione basandoti sulla descrizione e sui dati RAG.

## 1. CRITERI DI VALUTAZIONE
* **LIVELLO MINIMO (Accessibilità - Flessibile):** Qual è l'età minima per comprendere il concetto espresso?
    * Se il tema è **universale/divulgativo**, il minimo deve essere basso (es. 8), anche se il RAG propone capitoli accademici.
    * Se il tema è **tecnico/propedeutico** (es. crisi della Scolastica, logica aristotelica), il minimo deve rispecchiare la soglia d'ingresso dei capitoli pertinenti (es. 10).
* **LIVELLO MASSIMO (Potenziale - Rigido e Ancorato):** Fino a che livello si spinge il materiale a disposizione?
    * **REGOLA DEL TETTO:** Il livello Massimo della lezione NON DEVE MAI superare il valore massimo indicato dai capitoli RAG pertinenti. 
    * Se i capitoli pertinenti sul Medioevo o su un certo autore si fermano a 10, **il tuo massimo deve essere 10**. Non inventare estensioni teoriche (es. fino a 12) basandoti sulla presunta "complessità intrinseca" o sulle "implicazioni future" del tema. Il RAG riflette i veri limiti del curriculum scolastico.

## 2. FILTRO DI PERTINENZA
* Usa solo i capitoli che parlano dello **stesso argomento/periodo storico** della descrizione. 
* Scarta i capitoli fuori tema o che non riflettono l'argomento centrale.

## 3. GESTIONE FALLIMENTO (STIMA AUTONOMA)
* Applica la stima autonoma SOLO se non esiste nemmeno un capitolo pertinente. Se i capitoli ci sono, devi usare i loro dati numerici come tetto massimo.

## 3. FORMATO OUTPUT (JSON)
{
  "livello_min_consigliato": int,
  "livello_max_consigliato": int,
  "motivazione": "Spiega brevemente la scelta del minimo (accessibilità) e del massimo (potenziale) in relazione ai capitoli usati.",
  "fonte_livello": "capitoli_recuperati" O "stima_autonoma"
}
"""

In [134]:
PROMPT_TEMPLATE = """
### DESCRIZIONE LEZIONE
"{{descrizione_lezione}}"

### DATI RAG (RIFERIMENTI)
{{risultati_retriever}}

---
### ISTRUZIONI RAPIDE
1. Identifica il tema centrale.
2. Trova i capitoli RAG che parlano di quel tema.
3. Determina il **Minimo** in base a quanto è "aperto" o "tecnico" l'argomento.
4. Determina il **Massimo** prendendo il valore più alto dei capitoli pertinenti.
5. Rispondi in JSON.
"""

In [83]:
import os

In [41]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

In [42]:
from google import genai
from google.genai import types
import os
import json
from typing import List

In [43]:
GEMINI_MODEL = 'gemini-2.5-flash'
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [127]:
from pydantic import BaseModel, Field
from typing import Literal
class LivelloResult(BaseModel):
    livello_min_consigliato: int 
    livello_max_consigliato: int
    motivazione: str = Field(description = "Breve spiegazione del perché questi livelli sono stati scelti in base ai capitoli trovati")
    fonte_livello: Literal["capitoli_recuperati","stima_autonoma_su_descrizione"]
    #pertinenza_media_capitoli: Literal["Alta","Media","Bassa"]

In [135]:
def assign_min_max(query):
    dense_vector = next(dense_embedding_model.embed(query)) #next(dense_embedding_model.query_embed(query))
    sparse_vector = next(bm25_embedding_model.embed(query))#next(bm25_embedding_model.query_embed(query))
    late_interaction_vector = next(late_interaction_embedding_model.embed(query)) #next(late_interaction_embedding_model.query_embed(query))
    prefetch = [
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit=20,
            ),
            models.Prefetch(
                query=models.SparseVector(**sparse_vector.as_object()),
                using="bm25",
                limit=1,
            ),
        ]
    results = client.query_points(
             "rag_lezioni",
            prefetch=prefetch,
            query=late_interaction_vector,
            using="colbert",
            with_payload=True,
            limit=20,
    )
    
    points = results.points
    results = model.rerank(query, [point.payload['descr'] for point in points])

    results = [point.payload for point in points if point.payload['descr'] in [result['document'] for result in results[:10]]]
    prompt = PROMPT_TEMPLATE.replace('{{descrizione_lezione}}', query).replace('{{risultati_retriever}}', '\n'.join([f'{index}. Descrizione: {res["descr"]}\nTitolo: {res["titolo"]}\nLivello Minimo: {res["liv_min"]}\nLivello Massimo: {res["liv_max"]}' for index, res in enumerate(results)]))
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=content_list,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MESSAGE,
            temperature=0.3,
            response_mime_type='application/json',
            response_schema=LivelloResult)
    )

    return result.parsed


In [136]:
results_min = 0
results_max = 0
count = 0
for idx in range(0,100):
    # if df_test['Ordine di scuola min'].iloc[idx] == df_test['Ordine di scuola max'].iloc[idx]:
    #     continue
    
    print('Descrizione: ' + str(df_test['Descrizione'].iloc[idx]))
    print('True level min: ' + str(df_test['Ordine di scuola min'].iloc[idx]))
    print('True level max: ' + str(df_test['Ordine di scuola max'].iloc[idx]))
    print('-'*100)
    res = assign_min_max(df_test.Descrizione.iloc[idx])
    print('Predicted level min: ' + str(res.livello_min_consigliato))
    print('Predicted level max: ' + str(res.livello_max_consigliato))
    print('-'*100)

    print('Motivazione: ' + res.motivazione)
    print('fonte_livello: '+ res.fonte_livello)
    print('-'*100)
    results_min += 1 if (df_test['Ordine di scuola min'].iloc[idx] == res.livello_min_consigliato) else 0
    results_max += 1 if df_test['Ordine di scuola max'].iloc[idx] == res.livello_max_consigliato else 0
    



Descrizione: Per la rubrica "Leggiamo i filosofi", in questa videolezione il prof. Matteo Saudino, ideatore del canale YouTube <i>BarbaSophia</i>, legge e analizza un brano tratto da "Psicoanalisi" (1938) di Sigmund Freud, in cui il filosofo analizza due concetti importanti: il transfert e l'ambiguità della traslazione paziente-analista.
True level min: 12.0
True level max: 12.0
----------------------------------------------------------------------------------------------------
Predicted level min: 12
Predicted level max: 12
----------------------------------------------------------------------------------------------------
Motivazione: La lezione analizza concetti specifici e complessi della psicoanalisi freudiana (transfert, traslazione paziente-analista). I capitoli RAG pertinenti su Freud e la psicoanalisi indicano un livello minimo e massimo di 12, riflettendo la natura specialistica dell'argomento, tipica degli ultimi anni di scuola superiore.
fonte_livello: capitoli_recuperati
-

83